# A/B test

检测优惠券活动对于GMV的提升效果（精准分层 + 多重检验校正 + 置信区间 + 效应量）

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import hashlib
from scipy import stats

In [2]:
df = pd.read_csv('../../data/raw/online_retail_two/online_retail_II.csv')

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   Invoice      1067371 non-null  object 
 1   StockCode    1067371 non-null  object 
 2   Description  1062989 non-null  object 
 3   Quantity     1067371 non-null  int64  
 4   InvoiceDate  1067371 non-null  object 
 5   Price        1067371 non-null  float64
 6   Customer ID  824364 non-null   float64
 7   Country      1067371 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 65.1+ MB


In [4]:
df.describe()

,Quantity,Price,Customer ID
count,1.067371e+06,1.067371e+06,824364.000000
mean,9.938898e+00,4.649388e+00,15324.638504
std,1.727058e+02,1.235531e+02,1697.464450
min,-8.099500e+04,-5.359436e+04,12346.000000
25%,1.000000e+00,1.250000e+00,13975.000000
50%,3.000000e+00,2.100000e+00,15255.000000
75%,1.000000e+01,4.150000e+00,16797.000000
max,8.099500e+04,3.897000e+04,18287.000000


# 数据清洗

In [5]:
df.dropna(subset=['Customer ID'], inplace=True)
df = df[(df['Quantity']>0) & (df['Price']>0)]
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['amount'] = df['Quantity']*df['Price']
CurrentDate = df['InvoiceDate'].max()

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 805549 entries, 0 to 1067370
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      805549 non-null  object        
 1   StockCode    805549 non-null  object        
 2   Description  805549 non-null  object        
 3   Quantity     805549 non-null  int64         
 4   InvoiceDate  805549 non-null  datetime64[ns]
 5   Price        805549 non-null  float64       
 6   Customer ID  805549 non-null  float64       
 7   Country      805549 non-null  object        
 8   amount       805549 non-null  float64       
dtypes: datetime64[ns](1), float64(3), int64(1), object(4)
memory usage: 61.5+ MB


In [7]:
df.describe()

,Quantity,InvoiceDate,Price,Customer ID,amount
count,805549.000000,805549,805549.000000,805549.000000,805549.000000
mean,13.290522,2011-01-02 10:24:44.106814464,3.206561,15331.954970,22.026505
min,1.000000,2009-12-01 07:45:00,0.001000,12346.000000,0.001000
25%,2.000000,2010-07-07 12:08:00,1.250000,13982.000000,4.950000
50%,5.000000,2010-12-03 15:10:00,1.950000,15271.000000,11.850000
75%,12.000000,2011-07-28 13:05:00,3.750000,16805.000000,19.500000
max,80995.000000,2011-12-09 12:50:00,10953.500000,18287.000000,168469.600000
std,143.634088,NaN,29.199173,1696.737039,224.041928


In [8]:
df_customers = (
    df
    .groupby('Customer ID')
    .agg(
        r = ('InvoiceDate', lambda x: (CurrentDate- x.max()).days),
        f = ('Invoice', 'nunique'),
        m = ('amount', 'sum')
    )
    .reset_index()
)
df_customers['aov'] = df_customers['m']/df_customers['f']

# RFM客户分类

In [9]:
df_customers['r_score'] = pd.qcut(df_customers['r'], 2, labels=[2, 1]).astype(int)
df_customers['f_score'] = pd.qcut(df_customers['f'].rank(method='first'), 2, labels=[1, 2]).astype(int)
df_customers['m_score'] = pd.qcut(df_customers['m'], 2, labels=[1, 2]).astype(int)
df_customers['rfm_label'] = df_customers['r_score'].astype(str)+df_customers['f_score'].astype(str)+df_customers['m_score'].astype(str)
rfm_map = {
    '222': 'Champions',            # 重要价值用户
    '122': 'Loyal Customers',       # 重要保持用户
    '212': 'Potential Loyalists',   # 重要发展用户
    '112': 'Need Attention',        # 重要挽留用户
    '221': 'New Customers',         # 一般价值用户
    '121': 'Promising',             # 一般保持用户
    '211': 'About To Sleep',        # 一般发展用户
    '111': 'Lost Customers'         # 流失用户
}
df_customers['user_type'] = df_customers['rfm_label'].map(rfm_map)

# 哈希md5随机分组

In [10]:
df_customers.to_csv("../../data/processed/rfm_customer_analysis.csv", index=False, encoding='utf-8-sig')

In [11]:
salt = 'ab_test_coupon_2025'
df_customers['group'] = (
    (df_customers['Customer ID'].astype(str)+salt)
    .apply(lambda x: hashlib.md5(x.encode()).hexdigest())
    .apply(lambda x: (int(x, 16)%2/2)>=0.5).astype(int)
)

# SRM：样本均衡检验
SRM 卡方检验: 伯努利实验，各组样本服从二项分布，卡方统计量=∑(O_i -E_i)^2/E_i

In [12]:
control = df_customers[df_customers['group']==0]
test = df_customers[df_customers['group']==1]
n_control = len(control)
n_test = len(test)
observed = [n_control, n_test]
expected = [sum(observed)/2, sum(observed)/2]

chi_srm, p_srm = stats.chisquare(f_obs=observed, f_exp=expected)
print(f'对照组人数：{n_control}, 实验组人数：{n_test}')
print(f'srm chi-square: {chi_srm:.4f}, srm p:{p_srm:.4f}')

对照组人数：2903, 实验组人数：2975
srm chi-square: 0.8819, srm p:0.3477


p值大于0.05，不能拒绝原假设：即分流正常，样本比例=1：1

In [13]:
df_customers.head()

,Customer ID,r,f,m,aov,r_score,f_score,m_score,rfm_label,user_type,group
0,12346.0,325,12,77556.46,6463.038333,1,2,2,122,Loyal Customers,1
1,12347.0,1,8,5633.32,704.165000,2,2,2,222,Champions,1
2,12348.0,74,5,2019.40,403.880000,2,2,2,222,Champions,0
3,12349.0,18,4,4428.69,1107.172500,2,2,2,222,Champions,1
4,12350.0,309,1,334.40,334.400000,1,1,1,111,Lost Customers,1


# uplift 伪装增效
对不同的客户施加不同的虚拟活动效果

In [14]:
uplift_map = {
    "Champions": 0.05,
    "Loyal Customers": 0.08,
    "Potential Loyalists": 0.12,
    "New Customers": 0.15,
    "Promising": 0.10,
    "Need Attention": 0.07,
    "At Risk": 0.20,
    "About to Sleep": 0.18,
    "Lost Customers": 0.03
}
df_customers['final_m'] = df_customers['m'].copy()
for user_type, uplift in uplift_map.items():
    mask = (df_customers['user_type'] == user_type) & (df_customers['group'] == 1)
    df_customers.loc[mask, 'final_m'] *= (1+uplift)

# 整体检验

In [15]:
treatment = df_customers[df_customers['group']==1]['final_m']
control = df_customers[df_customers['group']==0]['final_m']
t_stat, p_2side = stats.ttest_ind(treatment, control, equal_var=False)
p_1side = p_2side/2

In [16]:
p_1side

np.float64(0.09934489320791819)

p值大于0.05， 不拒绝原假设

优惠券活动对整体客户的GMV促进效果并不显著，转向对于优惠券发给谁，发什么券，什么时候发，长期价值的检验

# 亚组检验（含多重检验校正）

In [17]:
num_tests = len(uplift_map) 
alpha = 0.05
alpha_corrected = alpha / num_tests 

def cohen_d(x, y):
    n1, n2 = len(x), len(y)
    if n1 < 2 or n2 < 2:
        return 0
    s1, s2 = x.var(), y.var()
    pooled_std = np.sqrt(((n1-1)*s1 + (n2-1)*s2) / (n1 + n2 - 2))
    return (x.mean() - y.mean()) / pooled_std if pooled_std != 0 else 0

def mean_ci(x, y, alpha=0.05):
    try:
        mean_diff = x.mean() - y.mean()
        n1, n2 = len(x), len(y)
        se = np.sqrt(x.var()/n1 + y.var()/n2)
        df = min(n1, n2) - 1
        t_val = stats.t.ppf(1 - alpha/2, df)
        return (round(mean_diff - t_val*se, 2), round(mean_diff + t_val*se, 2))
    except:
        return (None, None)

In [18]:
final_result = []

for user_type in uplift_map.keys():
    treat = df_customers[(df_customers['user_type'] == user_type) & (df_customers['group'] == 1)]['final_m']
    ctrl   = df_customers[(df_customers['user_type'] == user_type) & (df_customers['group'] == 0)]['final_m']
    
    nt, nc = len(treat), len(ctrl)
    
    if nt < 5 or nc < 5:
        print(f"\n{user_type} 样本量不足，跳过检验")
        continue

    # 检验方法选择
    if nt > 30 and nc > 30:
        stat, p_2side = stats.ttest_ind(treat, ctrl, equal_var=False)
        method = "t-test"
    else:
        # 小样本正态检验
        _, pnc = stats.shapiro(ctrl.sample(min(5000, nc))) if nc >=3 else (0, 1)
        _, pnt = stats.shapiro(treat.sample(min(5000, nt))) if nt >=3 else (0, 1)
        if pnc > 0.05 and pnt > 0.05:
            stat, p_2side = stats.ttest_ind(treat, ctrl, equal_var=False)
            method = "t-test (正态)"
        else:
            stat, p_2side = stats.ranksums(treat, ctrl)
            method = "Wilcoxon"

    p1 = p_2side / 2
    significant = p1 < alpha_corrected
    lift = (treat.mean() - ctrl.mean()) / ctrl.mean() if ctrl.mean() != 0 else 0
    d = cohen_d(treat, ctrl)
    ci_low, ci_high = mean_ci(treat, ctrl)

    final_result.append({
        "user_type": user_type,
        "treat_n": nt,
        "ctrl_n": nc,
        "treat_mean": round(treat.mean(), 2),
        "ctrl_mean": round(ctrl.mean(), 2),
        "lift": round(lift*100, 2),
        "cohen_d": round(d, 3),
        "p1": round(p1, 4),
        "significant_corrected": significant,
        "95CI_low": ci_low,
        "95CI_high": ci_high,
        "method": method
    })
    
res_df = pd.DataFrame(final_result)
print("\n" + "="*80)
print("A/B 测试分层最终结果")
print("="*80)
print(res_df.to_string(index=False))


At Risk 样本量不足，跳过检验

About to Sleep 样本量不足，跳过检验

A/B 测试分层最终结果
          user_type  treat_n  ctrl_n  treat_mean  ctrl_mean   lift  cohen_d     p1  significant_corrected  95CI_low  95CI_high method
          Champions      903     911     8560.35    6697.74  27.81    0.071 0.0669                  False   -573.53    4298.75 t-test
    Loyal Customers      337     312     3466.53    3211.98   7.92    0.042 0.2952                  False   -675.61    1184.70 t-test
Potential Loyalists      113      93     1527.20    3694.43 -58.66   -0.185 0.1167                  False  -5756.22    1421.76 t-test
      New Customers      117     128      707.66     632.22  11.93    0.363 0.0028                   True     21.99     128.90 t-test
          Promising      122     109      677.15     575.72  17.62    0.514 0.0001                   True     49.82     153.04 t-test
     Need Attention      139     131     2265.04    2018.32  12.22    0.061 0.3083                  False   -727.17    1220.60 t-test
 

优惠券活动对于促进一般价值用户，一般保持用户的消费促进效果比较明显，对于重要价值用户，重要保持用户，重要发展用户，重要挽留客户, 流失客户的消费促进效果不显著

In [19]:
res_df.to_csv("../../data/processed/ab_test_result.csv", index=False, encoding="utf-8-sig")

In [20]:
print("\n")
print("="*60)
print("                 A/B 测试业务决策结论")
print("="*60)
print("1. 全量发券对 GMV 无显著提升，不建议全量投放。")
print("2. 经 Bonferroni 校正后，以下人群优惠券效果显著：")
for _, row in res_df.iterrows():
    if row['significant_corrected']:
        print(f"   ✅ {row['user_type']} | 提升 {row['lift']}% | 95%CI [{row['95CI_low']}, {row['95CI_high']}]")
print("\n3. 其余人群效果不显著，不建议发券。")
print("4. 落地策略：精准定向高潜人群，停止全量投放。")
print("="*60)



                 A/B 测试业务决策结论
1. 全量发券对 GMV 无显著提升，不建议全量投放。
2. 经 Bonferroni 校正后，以下人群优惠券效果显著：
   ✅ New Customers | 提升 11.93% | 95%CI [21.99, 128.9]
   ✅ Promising | 提升 17.62% | 95%CI [49.82, 153.04]

3. 其余人群效果不显著，不建议发券。
4. 落地策略：精准定向高潜人群，停止全量投放。
